# AIRL applied workflow

A taxi operator learns a state-only zone-value reward from observed routes, then re-solves that reward after eastbound congestion changes the transition system.

In [1]:
from pathlib import Path
import pickle
import tempfile

import jax.numpy as jnp
import numpy as np
import econirl
from econirl import AIRL, Panel, RewardSpec, Trajectory
from econirl.core.bellman import SoftBellmanOperator
from econirl.core.solvers import value_iteration
from econirl.core.types import DDCProblem

module_path = Path(econirl.__file__).resolve()
print(f"Installed package import: {'site-packages' in str(module_path)}")
print(f"econirl version: {econirl.__version__}")

Installed package import: True
econirl version: 0.0.10


In [2]:
grid = 3
n_states, n_actions = grid * grid, 4
moves = ((-1, 0), (0, 1), (1, 0), (0, -1))
transitions = np.zeros((n_actions, n_states, n_states))
for action, (dr, dc) in enumerate(moves):
    for state in range(n_states):
        row, col = divmod(state, grid)
        next_row = min(max(row + dr, 0), grid - 1)
        next_col = min(max(col + dc, 0), grid - 1)
        transitions[action, state, next_row * grid + next_col] = 1.0
features = np.array([
    [-((abs(row) + abs(col - 2)) / 4), float((row, col) == (1, 1))]
    for row in range(grid) for col in range(grid)
])
true_reward = features @ np.array([2.0, -1.0])
reward_matrix = np.repeat(true_reward[:, None], n_actions, axis=1)
problem = DDCProblem(n_states, n_actions, discount_factor=0.9)
oracle = value_iteration(SoftBellmanOperator(problem, jnp.asarray(transitions)), jnp.asarray(reward_matrix))
print(f"Transition shape: {transitions.shape}")
print(f"Feature rank: {np.linalg.matrix_rank(features)}/{features.shape[1]}")

Transition shape: (4, 9, 9)
Feature rank: 2/2


In [3]:
rng = np.random.default_rng(20260816)
trajectories = []
for individual in range(120):
    state = int(rng.integers(n_states))
    states, actions, next_states = [], [], []
    for _ in range(30):
        action = int(rng.choice(n_actions, p=np.asarray(oracle.policy[state])))
        successor = int(rng.choice(n_states, p=transitions[action, state]))
        states.append(state); actions.append(action); next_states.append(successor)
        state = successor
    trajectories.append(Trajectory(jnp.array(states), jnp.array(actions), jnp.array(next_states), individual))
train = Panel(trajectories[:90])
test = Panel(trajectories[90:])
pairs = np.unique(np.stack([np.asarray(train.get_all_states()), np.asarray(train.get_all_actions())], axis=1), axis=0)
print(f"Train observations: {train.num_observations}")
print(f"Test observations: {test.num_observations}")
print(f"State-action coverage: {len(pairs)}/{n_states * n_actions}")

Train observations: 2700
Test observations: 900
State-action coverage: 32/36


In [4]:
spec = RewardSpec.state_dependent(jnp.asarray(features), ['downtown_access', 'congestion_zone'], n_actions)
model = AIRL(
    n_states=n_states, n_actions=n_actions, discount=0.9,
    max_rounds=60, min_rounds=40, discriminator_steps=3,
    compute_se=True, n_bootstrap=3, seed=17, se_seed=23,
).fit(train, transitions=transitions, reward=spec)
test_states = np.asarray(test.get_all_states())
test_actions = np.asarray(test.get_all_actions())
test_probs = model.predict_proba(test_states)
test_log_loss = -np.log(test_probs[np.arange(len(test_actions)), test_actions] + 1e-12).mean()
print(f"Converged: {model.converged_}")
print(f"Holdout log loss: {test_log_loss:.4f}")
print(f"Bootstrap draws: {model.bootstrap_.n_successful}/{model.bootstrap_.n_requested}")

Converged: True
Holdout log loss: 1.1403
Bootstrap draws: 3/3


In [5]:
print(model.summary())

Estimator
AIRL (state-only tabular adversarial IRL)

Data
Observations: 2700
Individuals: 90
State coverage: 1.000

Model
States: 9
Actions: 4
Discount: 0.9
Reward: state-only linear basis

Pre-estimation checks
Transition orientation: (n_actions, n_states, n_states)
Reward feature rank: 2

Fit
Converged: True
Rounds: 40
Final discriminator loss: 1.52318

Outcome
Log likelihood: -3110.073791
Recovered object: centered state reward and induced policy

Uncertainty
Trajectory bootstrap: 3/3 draws

Limitations
Transfer interpretation requires state-only rewards and AIRL's decomposability conditions.
Raw adversarial weights are not structural coefficients.


In [6]:
changed = transitions.copy()
for state in (0, 3, 6):
    intended = int(np.argmax(transitions[1, state]))
    changed[1, state] = 0.0
    changed[1, state, intended] = 0.25
    changed[1, state, state] += 0.75
result = model.counterfactual(transitions=changed, description='eastbound congestion')
policy_change = 0.5 * np.abs(np.asarray(result.policy_change)).sum(axis=1)
print(f"Mean policy change TV: {policy_change.mean():.4f}")
print(f"Largest zone policy change TV: {policy_change.max():.4f}")

Mean policy change TV: 0.0544
Largest zone policy change TV: 0.1231


In [7]:
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / 'airl.pkl'
    path.write_bytes(pickle.dumps(model))
    restored = pickle.loads(path.read_bytes())
    prediction_gap = np.max(np.abs(restored.predict_proba(np.arange(n_states)) - model.predict_proba(np.arange(n_states))))
    restored_change = restored.counterfactual(transitions=changed).counterfactual_policy
    counterfactual_gap = np.max(np.abs(np.asarray(restored_change) - np.asarray(result.counterfactual_policy)))
print(f"Prediction reload gap: {prediction_gap:.1e}")
print(f"Counterfactual reload gap: {counterfactual_gap:.1e}")

Prediction reload gap: 0.0e+00
Counterfactual reload gap: 0.0e+00
